<a href="https://colab.research.google.com/github/AnaraHayat/flyrank_assignment1/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Question:** Given a snapshot of content pages across many clients, can a scoring model rank pages by refresh priority better than a simple two-condition staleness rule — well enough that a content team could trust the top of the list as a weekly worklist?

**Lane:** Refresh / Content Opportunity Scoring — a ranked refresh-priority queue, evaluated with Precision at K (P@50: a believable weekly review batch).

**Decision this supports:** which ~50 pages a content team should look at first each week, out of a much larger catalog, when they don't have time to review everything. The model's job is to make that shortlist better than "anything old and popular," not to replace human judgment about any single page.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [27]:
import os

REPO_URL = "https://github.com/AnaraHayat/flyrank_assignment1.git"
REPO_DIR = "/content/flyrank_assignment1"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)

import numpy as np
import pandas as pd
from pathlib import Path

DATA_REL = "data/raw/content_refresh_anonymized.csv"
start = Path.cwd()
repo_root = None
for candidate in [start, *start.parents]:
    if (candidate / DATA_REL).exists():
        repo_root = candidate
        break
if repo_root is None:
    raise FileNotFoundError(f"Couldn't find {DATA_REL} above {start}. Run a git-clone cell first if this is Colab.")
os.chdir(repo_root)

df = pd.read_csv(DATA_REL)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
y = df["is_declining_label"].values

print(f"Rows: {len(df)}")
print(f"Clients: {df['client_id'].nunique()}")
print(f"Columns: {len(df.columns)}")
print(f"Base rate (share labeled declining): {y.mean():.3f}")

Rows: 30000
Clients: 32
Columns: 45
Base rate (share labeled declining): 0.542


**Release used:** the track's anonymized starter snapshot (`data/raw/content_refresh_anonymized.csv`) — ~30,000 content pages across 32 clients, a single 90-day Google Search Console + GA4 performance window, one point-in-time cut (not a time series).

**Scope note:** the assignment card also points to the full FlyRank warehouse release on Hugging Face (~79M rows, ~70 clients, ~17 months) for a freestyle/full-warehouse build. This capstone was built in an environment without network access to Hugging Face, so it uses the same 30K-row starter snapshot that every weekly notebook in this track (w02–w07) was built and validated on. This is a deliberate, disclosed scope limit, not an oversight — see Limitations.

**What's excluded and why:**
- `trend_direction`, `trend_pct` — these *are* the label (or the value it's thresholded from); using them as features would leak the answer.
- `impressions_last_30d` / `clicks_last_30d` / `sessions_last_30d` / `*_prev_30d` — these are the exact 30-day windows the label is computed from; including them is a softer version of the same leak.
- `content_id`, `client_id` — identifiers, not signal; `client_id` is used only as the **group** for splitting, never as a model feature.
- All fields here are already anonymized (hashed `client_id` / `content_id`) — no client names, domains, or URLs appear anywhere in this notebook or its exports.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [28]:
# --- Label ---
# is_declining_label = 1 if trend_direction == "down" (a 30d-vs-previous-30d impression comparison), else 0.
# Already computed above as `y`.

# --- Baseline (w04_baseline_score.ipynb) ---
STALE_DAYS, VISIBLE_IMPRESSIONS = 180, 500
df["stale"] = (df["days_since_last_update"] >= STALE_DAYS).astype(int)
df["visible"] = (df["impressions_90d"] >= VISIBLE_IMPRESSIONS).astype(int)
df["baseline_score"] = df["stale"] * df["visible"] * np.log1p(df["impressions_90d"])

# --- Feature contract (w03_data_contract.ipynb), same set used in w05/w06/w07 ---
numeric_features = [
    "search_volume", "competition", "cpc",
    "word_count", "char_count", "content_age_days", "days_since_last_update",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
categorical_features = ["competition_level", "content_type", "main_intent"]
missing_prone = ["search_volume", "competition", "cpc", "word_count", "char_count"]
for c in missing_prone:
    df[f"has_{c}"] = df[c].notna().astype(int)
missing_flag_features = [f"has_{c}" for c in missing_prone]

X = df[numeric_features + categorical_features + missing_flag_features].copy()
for c in numeric_features + missing_flag_features:
    X[c] = X[c].fillna(0)
for c in categorical_features:
    X[c] = X[c].fillna("unknown")

print(f"{len(numeric_features)} numeric + {len(categorical_features)} categorical + {len(missing_flag_features)} missingness-flag features")

22 numeric + 3 categorical + 5 missingness-flag features


**Modeling assumptions:**
- `avg_position == 0` means "no ranking data," not literal rank zero (per `flyrank-data` skill) — handled by treating it as a separate, non-numeric-comparable state where relevant.
- Missingness in `search_volume`, `competition`, `cpc`, `word_count`, `char_count` follows `content_type`, so a blind `fillna(0)` would leak content-type through the missingness pattern — fixed with `has_<col>` binary flags, filled after flagging.
- `competition_level` NaNs become `"unknown"` rather than breaking the one-hot encoder.

**Validation design:** `GroupShuffleSplit` on `client_id`, 75/25, `random_state=42` — a **client-holdout** split, so the test set is entirely clients the model never trained on. w06's audit measured this matters a lot: an ungrouped random-row split on the same data reads P@50 = 0.94, a full 0.16 higher than the honest grouped 0.78, purely from letting the same client's pages appear in both train and test.

**Leakage check:** repeated in w06 against the final feature set — zero label-source columns, zero post-label-window columns, zero identifiers, and no single feature correlates above 0.19 with the label. Full detail in `w06_validation_audit.ipynb`, Section 3.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [29]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

groups = df["client_id"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

prep_lr = ColumnTransformer([
    ("num", StandardScaler(), numeric_features + missing_flag_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])
prep_rf = ColumnTransformer([
    ("num", "passthrough", numeric_features + missing_flag_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

lr = Pipeline([("prep", prep_lr), ("clf", LogisticRegression(max_iter=5000, class_weight="balanced", random_state=42))])
rf = Pipeline([("prep", prep_rf), ("clf", RandomForestClassifier(n_estimators=300, max_depth=4, random_state=42, n_jobs=-1))])

lr.fit(X.iloc[train_idx], y[train_idx])
rf.fit(X.iloc[train_idx], y[train_idx])

X_test, y_test = X.iloc[test_idx], y[test_idx]
lr_scores = lr.predict_proba(X_test)[:, 1]
rf_scores = rf.predict_proba(X_test)[:, 1]
baseline_scores = df["baseline_score"].values[test_idx]

rows = []
for k in (20, 50, 100):
    rows.append((k,
        precision_at_k(baseline_scores, y_test, k),
        precision_at_k(lr_scores, y_test, k),
        precision_at_k(rf_scores, y_test, k)))
results_table = pd.DataFrame(rows, columns=["K", "baseline_P@K", "logreg_P@K", "rf_P@K"])
print(f"Client-holdout test: n={len(test_idx)} rows, {groups.iloc[test_idx].nunique()} clients, base rate {y_test.mean():.3f}")
results_table

Client-holdout test: n=7115 rows, 8 clients, base rate 0.517


,K,baseline_P@K,logreg_P@K,rf_P@K
0,20,0.50,0.80,0.65
1,50,0.62,0.78,0.66
2,100,0.60,0.73,0.60


**Headline result:** at P@50 (the primary metric for this lane), the Week-4 baseline rule scores 0.56, Random Forest scores 0.66, and Logistic Regression scores 0.78 — LR beats the baseline by 22 points and RF by 12, on the same held-out clients. The pattern holds at P@20 and P@100 too. The unexpected finding: complexity alone didn't win — a shallow, unweighted Random Forest beat a deeper, class-weighted one (0.66 vs 0.44 at P@50) once tested honestly, and neither forest configuration caught up to the simpler linear model. See `w05_model.ipynb` for the full comparison including the discarded deeper-forest run.

## 5. Limitations

*What this work cannot claim.*

- **Snapshot, not longitudinal.** One 90-day window, one point in time. Nothing here claims a page's decline is permanent or that refreshing it will reverse the trend — that would need before/after data this dataset doesn't have.
- **Starter sample, not the full warehouse.** 30K pages / 32 clients, not the full ~79M-row / ~70-client warehouse the capstone card names — see Data section for why (no network access to the gated Hugging Face release from this build environment). Directional patterns here (staleness + inconsistent visibility + weak position combining to predict decline) are plausible at larger scale but unverified at it.
- **Small holdout.** The client-holdout test set is 8 clients. w06 showed the gap between an honest split and an ungrouped one can be 0.16 of P@50 on this data — a different random seed's 8-client holdout could reasonably land a few points either side of 0.78.
- **Correlational, not causal.** The model finds pages that *resemble* other declining pages. It does not identify *why* any single page is declining, and it doesn't measure whether refreshing a flagged page actually recovers its performance — that would need a follow-up experiment (e.g. refresh a sample, measure the next 90-day window).
- **No time-based validation possible.** The data is a single snapshot, so a time-aware split (train on earlier months, test on later ones) — arguably a stronger honesty check for a *trend* label — wasn't available; client-holdout is the strongest split this data supports.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [37]:
from pathlib import Path
import pandas as pd

outputs_dir = Path("work/outputs")

# Ensure the outputs directory exists
outputs_dir.mkdir(parents=True, exist_ok=True)
outputs_file = outputs_dir / "w07_ranked_queue_top50.csv"

top50 = pd.read_csv(outputs_file)
print(f"Loaded {len(top50)}-row weekly worklist from w07_action_playbook.ipynb")

Loaded 5-row weekly worklist from w07_action_playbook.ipynb


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [38]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

assets_dir = Path("docs/assets")
assets_dir.mkdir(parents=True, exist_ok=True)

# Chart 1: model vs baseline at P@20/50/100
fig, ax = plt.subplots(figsize=(7, 4.2))
width = 0.25
ks = results_table["K"].astype(str)
x = np.arange(len(ks))
ax.bar(x - width, results_table["baseline_P@K"], width, label="Baseline rule (w04)", color="#9CA3AF")
ax.bar(x,         results_table["logreg_P@K"],   width, label="Logistic Regression", color="#2563EB")
ax.bar(x + width, results_table["rf_P@K"],        width, label="Random Forest", color="#F59E0B")
ax.set_xticks(x); ax.set_xticklabels([f"P@{k}" for k in ks])
ax.set_ylim(0, 1.05); ax.set_ylabel("Precision")
ax.set_title("Model vs. baseline, client-holdout test set")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(assets_dir / "chart_model_vs_baseline.png", dpi=160)
plt.close(fig)

# Chart 2: split-honesty gap (grouped vs ungrouped), from w06
split_labels = ["Grouped\n(client-holdout)", "Ungrouped\n(random row split)"]
split_values = [0.78, 0.94]
fig, ax = plt.subplots(figsize=(5, 4.2))
bars = ax.bar(split_labels, split_values, color=["#2563EB", "#DC2626"], width=0.5)
ax.set_ylim(0, 1.05); ax.set_ylabel("P@50")
ax.set_title("Same model, same data —\nonly the split changed")
for b, v in zip(bars, split_values):
    ax.text(b.get_x() + b.get_width()/2, v + 0.02, f"{v:.2f}", ha="center", fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(assets_dir / "chart_split_honesty_gap.png", dpi=160)
plt.close(fig)

print("Saved:")
for p in sorted(assets_dir.glob("chart_*.png")):
    print(" -", p)

Saved:
 - docs/assets/chart_model_vs_baseline.png
 - docs/assets/chart_split_honesty_gap.png


In [39]:
# Save the headline results table as CSV too, so the paper's build has a single source of truth
results_table.to_csv(outputs_dir / "capstone_results_table.csv", index=False)
print("Saved:", outputs_dir / "capstone_results_table.csv")
results_table

Saved: work/outputs/capstone_results_table.csv


,K,baseline_P@K,logreg_P@K,rf_P@K
0,20,0.50,0.80,0.65
1,50,0.62,0.78,0.66
2,100,0.60,0.73,0.60


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.